In [1]:
import pandas as pd
import requests

In [2]:
# USGS Earthquake API endpoint
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Query parameters to retrieve 1 earthquake in GeoJSON format
params = {
    "format": "geojson",
    "eventtype": "earthquake",
    "minmagnitude": 2.0,
    "starttime": "2026-10-01T00:00:00",
    "endtime": "2026-10-31T23:59:59",
    "limit": 20000
}

In [3]:
# Send GET request
response = requests.get(url, params=params, timeout=10)
data = response.json()
response

<Response [200]>

In [4]:
# Read the GeoJSON features into a pandas DataFrame
df = pd.json_normalize(data["features"])

# Extract longitude, latitude, and depth from geometry.coordinates
df[["longitude", "latitude", "depth_km"]] = pd.DataFrame(
    df["geometry.coordinates"].tolist(), index=df.index
)

# Clean up column prefixes (e.g., 'properties.mag' -> 'mag')
df.columns = [col.replace("properties.", "") for col in df.columns]

df["datetime"] = pd.to_datetime(df["time"], unit="ms", utc=True)

print(f"Retrieved {len(df)} earthquakes.")
df.head()

Retrieved 3849 earthquakes.


,type,id,mag,place,time,updated,tz,url,detail,felt,...,gap,magType,type,title,geometry.type,geometry.coordinates,longitude,latitude,depth_km,datetime
0,Feature,us7000tule,4.80,"199 km NW of Port-Olry, Vanuatu",1793491075354,1793492226040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,110,mb,earthquake,"M 4.8 - 199 km NW of Port-Olry, Vanuatu",Point,"[165.6627, -13.8748, 10]",165.662700,-13.874800,10.00,2026-10-31 23:57:55.354000+00:00
1,Feature,uu80162266,2.18,"62 km SW of Dugway, Utah",1793490900380,1793647273230,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,134,ml,earthquake,"M 2.2 - 62 km SW of Dugway, Utah",Point,"[-113.2995, 39.8575, 4]",-113.299500,39.857500,4.00,2026-10-31 23:55:00.380000+00:00
2,Feature,aka2026vpmcdb,2.00,"62 km SE of Kokhanok, Alaska",1793490188184,1793607665202,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,56,ml,earthquake,"M 2.0 - 62 km SE of Kokhanok, Alaska",Point,"[-153.998, 59.041, 111.9]",-153.998000,59.041000,111.90,2026-10-31 23:43:08.184000+00:00
3,Feature,hv75066981,2.20,"7 km SW of Volcano, Hawaii",1793489463070,1793516180040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,51,ml,earthquake,"M 2.2 - 7 km SW of Volcano, Hawaii",Point,"[-155.2785, 19.3923333333333, 1.77]",-155.278500,19.392333,1.77,2026-10-31 23:31:03.070000+00:00
4,Feature,nc75460612,2.76,"52 km W of Petrolia, CA",1793488907920,1794716207040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,...,238,md,earthquake,"M 2.8 - 52 km W of Petrolia, CA",Point,"[-124.892833333333, 40.3565, 16.61]",-124.892833,40.356500,16.61,2026-10-31 23:21:47.920000+00:00


In [5]:
# 1. Convenience sample: first 100 rows returned by the API's default sort order (most recent first)
convenience_df = df.iloc[:100].copy()

In [6]:
# 2. Simple Random Sample (SRS): n = 100 with random_state = 42
srs_df = df.sample(n=100, random_state=42).copy()

In [7]:
# 3. Compare mean magnitude and time span of both samples (x-bar) against the population (mu)
def summarize_dataset(data, label):
    mean_mag = data["mag"].mean()
    start_dt = data["datetime"].min()
    end_dt = data["datetime"].max()
    time_span = end_dt - start_dt
    return {
        "Dataset": label,
        "Count (n)": len(data),
        "Mean Magnitude": round(mean_mag, 4),
        "Earliest Event (UTC)": start_dt,
        "Latest Event (UTC)": end_dt,
        "Time Span": time_span
    }

In [8]:
summary_df = pd.DataFrame([
    summarize_dataset(df, "Population (mu)"),
    summarize_dataset(convenience_df, "Convenience Sample (first 100)"),
    summarize_dataset(srs_df, "Simple Random Sample (n=100, seed=42)")
])

mu_mag = df["mag"].mean()
summary_df["Mag Diff from mu"] = (summary_df["Mean Magnitude"] - mu_mag).round(4)

summary_df

,Dataset,Count (n),Mean Magnitude,Earliest Event (UTC),Latest Event (UTC),Time Span,Mag Diff from mu
0,Population (mu),3849,3.3434,2026-10-01 00:00:21.945000+00:00,2026-10-31 23:57:55.354000+00:00,30 days 23:57:33.409000,0.0000
1,Convenience Sample (first 100),100,3.0002,2026-10-31 01:21:36.013000+00:00,2026-10-31 23:57:55.354000+00:00,0 days 22:36:19.341000,-0.3432
2,"Simple Random Sample (n=100, seed=42)",100,3.4098,2026-10-01 03:46:01.130000+00:00,2026-10-31 21:19:43.482000+00:00,30 days 17:33:42.352000,0.0664


### The reason that the convineance sample distorts the estimate is beacause there are much less datapoints and all the datapoints are clumped together at the start.